# Milestone 1: Data Processing, EDA and Visualization

Run the cells from top to bottom with `notebooks/` as the working directory. Functions are imported from `src/preprocessing.py`. The NLP setup downloads the original NLTK resources, and the save step writes the processed CSVs. Outputs have been cleared during reorganization; run cells to generate results.

## 1. Setup

### Import libraries and project functions

In [ ]:
# Import libraries and reusable project functions.
import ast
import os
import sys
from collections import Counter
from functools import partial
from pathlib import Path

import certifi
import matplotlib.pyplot as plt
import nltk
import pandas as pd
import seaborn as sns

# Run this notebook with notebooks/ as the working directory.
sys.path.insert(0, str(Path.cwd().parent))
from src.preprocessing import (
    extract_text_from_file,
    extract_resume_sections,
    clean_text,
    prepare_nlp,
    preprocess_nlp,
)


## 2. Data & Preprocessing

### Load the sample resumes and job descriptions

In [ ]:
cv = pd.read_csv("../data/sample/small_cv.csv")
jobs = pd.read_csv("../data/sample/small_jobs.csv")

print(cv.shape)
print(jobs.shape)


### Document loader and parsing limitations

In [ ]:
print("Loader ready. Note on parsing challenges: Complex PDFs with two-column layouts or nested tables often read out of order, jumbling Experience and Education dates.")


### Preview a resume and job description

In [ ]:
print("RESUME:")
print(cv.loc[0, "resume_text"])

print("\nJOB DESCRIPTION:")
print(jobs.loc[0, "job_description"])


### Inspect columns and data types

In [ ]:
# Check columns, data types, and non-null values

cv.info()

print("\n----------------------\n")

jobs.info()


### Check missing values

In [ ]:
# Check if there are any missing values

print("CV missing values:")
print(cv.isnull().sum())

print("\nJob missing values:")
print(jobs.isnull().sum())


### Check duplicate rows

In [ ]:
# Check if there are duplicate rows

print("CV duplicates:", cv.duplicated().sum())
print("Job duplicates:", jobs.duplicated().sum())


### Remove duplicate rows

In [ ]:
# Remove duplicate rows

cv = cv.drop_duplicates()
jobs = jobs.drop_duplicates()

print("CV shape:", cv.shape)
print("Jobs shape:", jobs.shape)


### Fill missing text values

In [ ]:
# Replace missing resume or job text with an empty string

cv["resume_text"] = cv["resume_text"].fillna("")
jobs["job_description"] = jobs["job_description"].fillna("")

print("Missing resume text:", cv["resume_text"].isnull().sum())
print("Missing job text:", jobs["job_description"].isnull().sum())


### Extract experience, education and skills sections

In [ ]:
cv['structured_sections'] = cv['resume_text'].apply(extract_resume_sections)
print("Structured sections extracted successfully.")


### Configure the SSL certificate path

In [ ]:
# Fix SSL certificate path


os.environ["SSL_CERT_FILE"] = certifi.where()


### Download the required NLTK resources

In [ ]:
# Download required NLTK resources

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


### Prepare the tokenizer, stop words and lemmatizer

In [ ]:
tokenizer, stop_words, lemmatizer = prepare_nlp()
preprocess_nlp = partial(
    preprocess_nlp,
    tokenizer=tokenizer,
    stop_words=stop_words,
    lemmatizer=lemmatizer,
)


### Clean resume and job text

In [ ]:
# Apply the cleaning function to resumes and job descriptions

cv["processed_resume"] = cv["resume_text"].apply(clean_text)

jobs["processed_job"] = jobs["job_description"].apply(clean_text)


### Compare original and cleaned text

In [ ]:
# Compare the text before and after preprocessing

print("ORIGINAL RESUME:")
print(cv.loc[0, "resume_text"][:500])

print("\nCLEANED RESUME:")
print(cv.loc[0, "processed_resume"][:500])

print("\nORIGINAL JOB:")
print(jobs.loc[0, "job_description"][:500])

print("\nCLEANED JOB:")
print(jobs.loc[0, "processed_job"][:500])


### Tokenize, remove stop words and lemmatize

In [ ]:
# Apply NLP preprocessing

cv["tokens"] = cv["processed_resume"].apply(preprocess_nlp)
jobs["tokens"] = jobs["processed_job"].apply(preprocess_nlp)


### Preview processed tokens

In [ ]:
# Display processed tokens

print("Resume Tokens:")
print(cv.loc[0, "tokens"][:50])

print("\nJob Tokens:")
print(jobs.loc[0, "tokens"][:50])


### Join tokens into model-ready text

In [ ]:
# Join processed tokens back into text for later modeling

cv["final_processed_resume"] = cv["tokens"].apply(
    lambda x: " ".join(x)
)

jobs["final_processed_job"] = jobs["tokens"].apply(
    lambda x: " ".join(x)
)


### Preview the final processed text

In [ ]:
# Display final processed text

print("FINAL RESUME TEXT:")
print(cv.loc[0, "final_processed_resume"][:500])

print("\nFINAL JOB TEXT:")
print(jobs.loc[0, "final_processed_job"][:500])


### Preprocessing Pipeline Documentation
1. **Raw Load:** Ingested PDF/DOCX/TXT via `PyPDF2` and `python-docx`.
2. **Section Extraction:** Applied Regex to isolate Experience, Education, and Skills.
3. **Cleaning:** Lowercased text, stripped URLs, and removed special characters.
4. **NLP Processing:** Tokenized, removed stopwords, and applied WordNetLemmatizer.

### Save the processed datasets

In [ ]:
# Save the final cleaned and preprocessed dataset
cv.to_csv('../data/processed/final_cleaned_cv.csv', index=False)
jobs.to_csv('../data/processed/final_cleaned_jobs.csv', index=False)
print("Final preprocessed datasets saved to CSV.")


## 3. EDA / Visualization

### Calculate resume word counts

In [ ]:
# Calculate the number of words in each resume

cv["resume_length"] = cv["final_processed_resume"].apply(
    lambda x: len(x.split())
)

cv[["target_position", "resume_length"]].head()


### Plot the resume length distribution

In [ ]:
# Visualize the distribution of resume lengths

plt.figure(figsize=(8, 5))

sns.histplot(cv["resume_length"], bins=10)

plt.title("Resume Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.show()


### Convert resume skill labels into lists

In [ ]:
# Convert resume skills from text into Python lists
cv["skills_list"] = cv["clean_skills"].apply(ast.literal_eval)

cv[["target_position", "skills_list"]].head()


### Count the most common resume skills

In [ ]:
# Count how often each skill appears in the resumes


all_resume_skills = []

for skills in cv["skills_list"]:
    all_resume_skills.extend(skills)

resume_skill_counts = Counter(all_resume_skills)

resume_top_skills = pd.DataFrame(
    resume_skill_counts.most_common(10),
    columns=["Skill", "Count"]
)

resume_top_skills


### Plot the top 10 resume skills

In [ ]:
# Visualize the most common skills in resumes

plt.figure(figsize=(10, 5))

sns.barplot(
    data=resume_top_skills,
    x="Count",
    y="Skill"
)

plt.title("Top 10 Skills in Resumes")
plt.xlabel("Count")
plt.ylabel("Skill")

plt.show()


### Convert job skill labels into lists

In [ ]:
# Convert job skills from text into Python lists

jobs["skills_list"] = jobs["clean_skills"].apply(ast.literal_eval)

jobs[["job_title", "skills_list"]].head()


### Count the most common required job skills

In [ ]:
# Count how often each skill is required by jobs

all_job_skills = []

for skills in jobs["skills_list"]:
    all_job_skills.extend(skills)

job_skill_counts = Counter(all_job_skills)

job_top_skills = pd.DataFrame(
    job_skill_counts.most_common(10),
    columns=["Skill", "Count"]
)

job_top_skills


### Plot the top 10 job skills

In [ ]:
# Visualize the most common required job skills

plt.figure(figsize=(10, 5))

sns.barplot(
    data=job_top_skills,
    x="Count",
    y="Skill"
)

plt.title("Top 10 Required Job Skills")
plt.xlabel("Count")
plt.ylabel("Skill")

plt.show()


### Count jobs in each category

In [ ]:
# Check the distribution of job categories

print(jobs["category"].value_counts())


### Plot the job category distribution

In [ ]:
# Visualize job categories

plt.figure(figsize=(10, 5))

sns.countplot(
    data=jobs,
    y="category",
    order=jobs["category"].value_counts().index
)

plt.title("Job Category Distribution")
plt.xlabel("Count")
plt.ylabel("Category")

plt.show()


### Inspect imbalance and top skills for each job domain

In [ ]:
# Investigate dataset imbalance

plt.figure(figsize=(10, 5))
sns.countplot(data=jobs, y='category', order=jobs['category'].value_counts().index)
plt.title("Job Category Distribution (Dataset Imbalance)")
plt.xlabel("Number of Postings")
plt.ylabel("Job Domain")
plt.show()

# TASK 3: Analyze top required skills per job domain
print("\n--- Top Required Skills per Job Domain ---")
domain_skills = jobs.groupby('category')['skills_list'].sum()

for domain, skills in domain_skills.items():
    print(f"\n{domain}:")
    skill_counts = Counter(skills)
    for skill, count in skill_counts.most_common(5):
        print(f"  - {skill}: {count}")


## 4. Findings

### EDA Report & Findings
* **Dataset Imbalance:** The dataset leans heavily toward HR and IT categories. Future iterations will require expanding the Finance and Sales data to prevent model bias.
* **Skill Frequency:** Broad skills like Python, Machine Learning, and adaptability dominate the requirements.
* **Parsing Bottlenecks:** PDF layouts with columns or non-standard bullet points cause the regex extractor to misclassify Experience and Education blocks.